In [ ]:
import time
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

FEATURES_CSV = "processed_features_conf_0.5_filled_metadata_no_collision.csv"
IMAGE_STATS_CSV = "image_stats_processed_final_grids_no_collision.csv"

OUTPUT_CSV = "final_dataset_no_collision.csv"

MERGE_KEYS = [
    "requested_latitude",
    "requested_longitude",
    "pano_id",
    "image_name",
]

# ============================================================
# LOAD
# ============================================================

t0 = time.time()

image_stats = pd.read_csv(
    IMAGE_STATS_CSV,
    engine="pyarrow"
)

print(f"Image stats read in {time.time()-t0:.1f}s")

t0 = time.time()

features = pd.read_csv(
    FEATURES_CSV,
    engine="pyarrow"
)

print(f"Features read in {time.time()-t0:.1f}s")

print(f"\nFeature rows     : {len(features):,}")
print(f"Image stats rows : {len(image_stats):,}")

# ============================================================
# REMOVE DUPLICATE COLUMNS
# ============================================================

duplicate_cols = [
    c
    for c in image_stats.columns
    if c not in MERGE_KEYS and c in features.columns
]

if duplicate_cols:
    print(f"\nDropping {len(duplicate_cols)} duplicate columns...")
    image_stats = image_stats.drop(columns=duplicate_cols)

# ============================================================
# MERGE
# ============================================================

t0 = time.time()

merged = features.merge(
    image_stats,
    on=MERGE_KEYS,
    how="inner",
    validate="one_to_one",
)

print(f"\nMerge completed in {time.time()-t0:.1f}s")

# ============================================================
# SUMMARY
# ============================================================

print("\n========================================")
print("MERGE SUMMARY")
print("========================================")

print(f"Rows after merge : {len(merged):,}")
print(f"Columns          : {len(merged.columns):,}")

print(f"Feature rows not matched    : {len(features) - len(merged):,}")
print(f"Image stats rows not matched: {len(image_stats) - len(merged):,}")

# ============================================================
# SAVE
# ============================================================

t0 = time.time()

merged.to_csv(
    OUTPUT_CSV,
    index=False,
)

print(f"\nSaved to {OUTPUT_CSV}")
print(f"Write time: {time.time()-t0:.1f}s")